# Solar challenge draw: example event
This is an example event to demonstrate seeding and the double elimination draw. One of the main features of this particular system is that it allows for the situation where both competitors “lose” by not running in a knockout race.

In [ ]:
# Export this using:
# jupyter-nbconvert ./literature/ExampleEvent.ipynb --no-prompt --no-input --to markdown

In [ ]:
# Libraries
from IPython.display import display, SVG, Markdown
import tkinter as tk
import ttkbootstrap as ttk
import ttkbootstrap.constants as ttkc
import subprocess
import sys
import os
from typing import List
import pandas as pd

In [ ]:
# Adding the programme to the path so that we can import it.
notebook_dir = os.path.dirname(os.getcwd())
 
# Add the project root to sys.path (one level up from notebooks/)  
project_root = os.path.abspath(os.path.join(notebook_dir, "SolarChallengeDraw"))
sys.path.append(project_root)  

In [ ]:
from knockout import KnockoutEvent, RoundId, RoundType
from car import Car
from knockout_race import Race
from knockout_sheet import KnockoutSheet
from knockout_sheet_elements import PrintNumberBoxFactory, InteractiveNumberBoxFactory
from save_load import Metadata

## Entrants
For this example, assume we have a small field 6 cars. The round-robin has been completed, and each car has been assigned a number of points (more points means a higher ranking). The cars are as follows in the table:

In [ ]:
cars = [
    Car(101, 1, "Flying fish", True, True, True, 1),
    Car(102, 1, "Curious cat", True, True, True, 1),
    Car(103, 1, "Hungry horse", True, True, True, 2),
    Car(104, 2, "Percy penguin", True, True, True, 2),
    Car(105, 3, "Munching mouse", True, True, True, 4),
    Car(106, 3, "Busy bee", True, True, True, 5)
]

In [ ]:
def cars_to_table(cars: List[Car]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Car ID": i.car_id,
                "Car name": i.car_name,
                "Points": i.points,
            }
            for i in cars
        ]
    )

display(Markdown(cars_to_table(cars).to_markdown(index=False)))

In [ ]:
event = KnockoutEvent.new_from_cars(cars=cars, name="Example event", max_auxilliary_races=5, grand_final_heats=3)
# event.print()

In [ ]:
def export_event(event:KnockoutEvent, filename:str) -> None:
    frame = ttk.Window()
    sheet = KnockoutSheet(frame, None, None)
    sheet.draw_canvas(event, metadata=Metadata(), numbers=PrintNumberBoxFactory(), show_seed=False)
    sheet.export("gs", filename, 297, 210, surpress_output=True)
    frame.destroy()

In [ ]:
def convert_pdf2svg(basename:str, surpress_output:bool=True) -> str:
    args = ["inkscape","--export-page=all","--export-type=svg","--pdf-poppler","-l",basename + ".pdf"]
    process = subprocess.Popen(
        args,
        stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL if surpress_output else None,
        stderr=subprocess.DEVNULL if surpress_output else None
    )
    process.wait()
    return basename + ".svg"

In [ ]:
def export_convert_display(event:KnockoutEvent, filename:str) -> None:
    export_event(event, filename)
    svg_filename = convert_pdf2svg(filename)
    display(SVG(svg_filename))

## Initial seed

The primary aim of seeding is to avoid a situation where the most competitive cars race and knock each other out in the first couple of rounds. As well as allowing competitors to practice and fine tune their cars, the round-robin provides a very rough indicator of where each car may place in the event, which is used for seeding.

Let's sort the table of cars by their round-robin points and assign an estimated finishing place. Ties between cars with the same number of points are settled at random. The race draw needs a power of two races, so let's add some placeholders to bring the number of entrants up to 8:

In [ ]:
ranking_table = cars_to_table(cars)
ranking_table.loc[-1] = ["-", "-", 0]
ranking_table.index += 1  # shifting index
ranking_table.loc[-1] = ["-", "-", 0]
ranking_table = ranking_table.sort_values("Points", ascending=False).reset_index(drop=True)
ranking_table.index += 1
ranking_table.index.name = "Estimated place"

display(Markdown(ranking_table.to_markdown(index=True)))

Now it's time to assign cars to races in round 1. Because there are 6 competitors (8 with the placeholders), round 1 will have 4 races. We want the theoretically most competitive cars - *Busy bee* (106) and *Munching mouse* (105) - to not race each other for as long as possible. We do this by putting them is separate halves of the round one bracket where they can only meet later on in the event. Likewise, we wish to delay races between the third and fourth places - *Percy penguin* (104) and *Hungry horse* (103) - so place these on separate halves as fair away from the first and second seeds as possible. This process continues until all competitors are assigned to a race. Note that this always ends up with the highest ranked seed racing the lowest, the second-highest racing the second lowest and so on.

Races where a competitor is running against a placeholder will become byes.

With all round 1 races assigned, we have our initial seed as shown below:

In [ ]:
export_convert_display(event, "./images/initial_seed.svg")

## Round P1

In this round:
- *Busy bee* (106) successfully moves under its own power to complete its bye (race 1). *Busy bee* (106) will next compete in race 7 in round P2.
- Despite being ranked lower, *Flying fish* (101) wins against *Percy penguin* (104). *Flying fish* (101) will next compete in race 7 against *Busy bee* (106). *Percy penguin* (104) is not out of the competition yet and will now enter the second chance draw.
- Despite having a bye, *Munching mouse* (105) is not able to move under its own power and did not run (DNR). *Munching mouse* (105) will enter the second chance rounds.
- Neither *Hungry horse* (103) nor *Curious cat* (102) were able to move in race 4 and received a DNR result. Because there is only a single position available in the second chance draw, auxiliary race 1 will be used to determine the car to progress to the second chance bracket.

In [ ]:
round_1 = event.get_round(RoundId(RoundType.WINNERS, 0))
round_1[0].set_winner(heat=0, car_number=106, auxilliary_manager=event.auxiliary_races)
round_1[1].set_winner(heat=0, car_number=101, auxilliary_manager=event.auxiliary_races)
round_1[2].set_winner(heat=0, car_number=Race.WINNER_DNR, auxilliary_manager=event.auxiliary_races)
round_1[3].set_winner(0, car_number=Race.WINNER_DNR, auxilliary_manager=event.auxiliary_races)
export_convert_display(event, "./images/round1.svg")